In [0]:
from delta.tables import DeltaTable
from pyspark.sql import functions as F
from pyspark.sql.window import Window

CATALOG_SCHEMA = "workspace.ecommerce_de"

# 1. Silver Orders (Deduplicated + Typed)
valid_orders = spark.table(f"{CATALOG_SCHEMA}.staging_valid_orders").withColumn(
    "created_at_ts", F.to_timestamp(F.col("created_at"))
)

order_win = Window.partitionBy("order_id").orderBy(
    F.col("created_at_ts").desc_nulls_last(),
    F.col("bronze_ingested_at").desc_nulls_last(),
)

silver_orders_df = (
    valid_orders.withColumn("rn", F.row_number().over(order_win))
    .filter(F.col("rn") == 1)
    .drop("rn", "order_amount")
    .withColumnRenamed("order_amount_clean", "order_amount")
    .withColumn("silver_processed_at", F.current_timestamp())
)

silver_orders_df.write.format("delta").mode("overwrite").option("overwriteSchema", "true").saveAsTable(
    f"{CATALOG_SCHEMA}.silver_orders"
)

# 2. Silver Payments (Parsed {amount, method, status} + Deduplicated + Typed)
raw_pay_str = F.regexp_replace(F.trim(F.col("payment")), r"^\{|\}$", "")
parts = F.split(raw_pay_str, ",")

valid_payments = (
    spark.table(f"{CATALOG_SCHEMA}.staging_valid_payments")
    .withColumn("payment_amount", F.trim(F.element_at(parts, 1)).cast("double"))
    .withColumn("payment_method", F.upper(F.trim(F.element_at(parts, 2))))
    .withColumn("payment_status", F.upper(F.trim(F.element_at(parts, 3))))
    .withColumn("event_ts", F.to_timestamp(F.col("event_timestamp")))
)

pay_win = Window.partitionBy("event_id").orderBy(
    F.col("event_ts").desc_nulls_last(),
    F.col("bronze_ingested_at").desc_nulls_last(),
)

silver_payments_df = (
    valid_payments.withColumn("rn", F.row_number().over(pay_win))
    .filter(F.col("rn") == 1)
    .drop("rn", "payment")
    .withColumn("silver_processed_at", F.current_timestamp())
)

silver_payments_df.write.format("delta").mode("overwrite").option("overwriteSchema", "true").saveAsTable(
    f"{CATALOG_SCHEMA}.silver_payments"
)

# 3. SCD Type 2 for Customers (dim_customers_silver)
cust_bronze = (
    spark.table(f"{CATALOG_SCHEMA}.bronze_customers")
    .withColumn("updated_ts", F.to_timestamp(F.col("updated_at")))
    .withColumn(
        "attr_hash",
        F.sha2(
            F.concat_ws(
                "||",
                F.coalesce(F.col("city"), F.lit("")),
                F.coalesce(F.col("loyalty_tier"), F.lit("")),
            ),
            256,
        ),
    )
)

cust_win = Window.partitionBy("customer_id").orderBy(
    F.col("updated_ts").desc_nulls_last(),
    F.col("bronze_ingested_at").desc_nulls_last(),
)

latest_cust_df = (
    cust_bronze.withColumn("rn", F.row_number().over(cust_win))
    .filter(F.col("rn") == 1)
    .drop("rn")
    .select(
        "customer_id",
        "city",
        "loyalty_tier",
        "attr_hash",
        F.col("updated_ts").alias("effective_from"),
    )
    .withColumn("effective_to", F.lit(None).cast("timestamp"))
    .withColumn("is_current", F.lit(True))
    .withColumn("silver_processed_at", F.current_timestamp())
)

dim_cust_table = f"{CATALOG_SCHEMA}.dim_customers_silver"

if not spark.catalog.tableExists(dim_cust_table):
    latest_cust_df.write.format("delta").mode("overwrite").saveAsTable(dim_cust_table)
else:
    target_dt = DeltaTable.forName(spark, dim_cust_table)
    target_df = target_dt.toDF()

    changed_keys_df = (
        latest_cust_df.alias("src")
        .join(
            target_df.filter(F.col("is_current") == True).alias("tgt"),
            on="customer_id",
            how="inner",
        )
        .filter(F.col("src.attr_hash") != F.col("tgt.attr_hash"))
        .select("src.customer_id")
        .distinct()
    )

    target_dt.alias("tgt").merge(
        changed_keys_df.alias("chg"),
        "tgt.customer_id = chg.customer_id AND tgt.is_current = true",
    ).whenMatchedUpdate(
        set={
            "is_current": F.lit(False),
            "effective_to": F.current_timestamp(),
            "silver_processed_at": F.current_timestamp(),
        }
    ).execute()

    existing_keys_df = target_df.select("customer_id").distinct()

    new_or_updated_df = (
        latest_cust_df.alias("src")
        .join(existing_keys_df.alias("ex"), on="customer_id", how="left_anti")
        .unionByName(
            latest_cust_df.alias("src").join(
                changed_keys_df.alias("chg"), on="customer_id", how="inner"
            )
        )
    )

    new_or_updated_df.write.format("delta").mode("append").saveAsTable(dim_cust_table)

print("✅ Day 4 complete: Silver Orders, Silver Payments & SCD Type 2 dim_customers_silver ready.")

In [0]:
%sql
SELECT 'silver_orders' AS tbl, COUNT(*) AS cnt FROM workspace.ecommerce_de.silver_orders
UNION ALL
SELECT 'silver_payments', COUNT(*) FROM workspace.ecommerce_de.silver_payments
UNION ALL
SELECT 'dim_customers_silver', COUNT(*) FROM workspace.ecommerce_de.dim_customers_silver;